In [ ]:
import random
import json
import csv
import requests
from tqdm import tqdm

In [20]:
class OpenRouter:
    def __init__(self, api_key: str, model: str = "meta-llama/llama-3.2-3b-instruct", title="openrouter-call"):
        self.api_key = api_key
        self.model = model
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "X-Title": title,
            "Content-Type": "application/json"
        }

    def chat(self, prompt: str, temperature=0.3, max_tokens=500):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        response = requests.post(self.url, headers=self.headers, json=payload)

        if response.status_code != 200:
            raise Exception(f"OpenRouter Error {response.status_code}: {response.text}")

        return response.json()['choices'][0]['message']['content'].strip()

In [25]:
# ✅ 提示词构造器
class PromptBuilder:
    def __init__(self, train_data_path: str, schema_csv_path = None, mode: str = 'zero-shot', shot_num: int = 0):
        self.mode = mode
        self.shot_num = shot_num

        with open(train_data_path, "r", encoding="utf-8") as f:
            self.examples = [json.loads(line.strip()) for line in f]
        
        self.schema_text = self._read_schema(schema_csv_path) if schema_csv_path else ""

    def _read_schema(self, schema_csv_path: str) -> str:
        schema_map = {}
        with open(schema_csv_path, newline='', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                table = row['Table Name'].strip()
                field = row[' Field Name'].strip()
                if table == '-' or field == '-':
                    continue
                if table not in schema_map:
                    schema_map[table] = []
                schema_map[table].append(field)
        return "\n".join([f"{table}({', '.join(fields)})" for table, fields in schema_map.items()])

    def build_prompt(self, input_question):
        instruction = (
            "You are a SQL query generator.\n"
            "Translate the given natural language question into a SQL query.\n"
            "Only output the SQL code, a single SQL query.\n\n"
            "Only use the schema provided.\n"
        )
        prompt=""
        prompt = instruction + f"Schema:\n{self.schema_text}\n\n"
        if self.mode != "zero-shot":
            sampled = random.sample(self.examples, min(self.shot_num, len(self.examples)))
            for ex in sampled:
                prompt += f"# Question: {ex['text']}\nSQL: {ex['sql']}\n\n"

        final_question = f"# Question: {input_question}\nSQL:"
        return instruction + prompt + final_question

In [ ]:
API_KEY = "YOUR_OPENROUTER_API_KEY"
MODEL = "meta-llama/llama-3.2-3b-instruct"
client = OpenRouter(api_key=API_KEY, model=MODEL)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line.strip()) for line in f]
# def clean_sql(sql):
#     return re.sub(r'\s+', ' ', sql.strip().lower())
# def evaluate(pred, refs):
#     pred = clean_sql(pred)
#     refs = [clean_sql(r) for r in refs]
#     return pred in refs

def evaluate(pred_sql: str, gold_sql: str) -> bool:
    return pred_sql.strip().rstrip(";") == gold_sql.strip().rstrip(";")

def run_debug(train_file, test_file, schema_csv_file, shot_type="many-shot", shot_count=3, n=5):
    builder = PromptBuilder(train_file, schema_csv_file, mode=shot_type, shot_num=shot_count)
    test_data = load_jsonl(test_file)[:n]

    correct = 0

    for idx, item in enumerate(tqdm(test_data)):
        prompt = builder.build_prompt(item["text"])
        print(f"\n--- Prompt #{idx+1} ---")
        # print(prompt)

        try:
            gen_sql = client.chat(prompt)
        except Exception as e:
            print(f"❌ Error on item {idx}: {e}")
            continue

        if not gen_sql.strip().endswith(";"):
            gen_sql += " ;"

        gold = item["sql"]
        match = evaluate(gen_sql, gold)
        correct += int(match)

        print(f"🧪 Q: {item['text']}")
        print(f"🔧 SQL_PRED: {gen_sql}")
        print(f"🎯 SQL_GOLD: {gold}")
        print(f"✅ MATCH: {match}")

    acc = correct / len(test_data)
    print(f"\n✅ [{shot_type} | {shot_count} shots] Accuracy: {acc:.4f} ({correct}/{len(test_data)})")


In [23]:
run_debug(
    train_file="generation_train.jsonl",
    test_file="generation_test.jsonl",
    schema_csv_file="atis-schema.csv",
    shot_type="many-shot",
    shot_count=5,
    n=3# 控制测试样本数
)

  0%|          | 0/3 [00:00<?, ?it/s]


--- Prompt #1 ---


 33%|███▎      | 1/3 [00:02<00:04,  2.48s/it]

🧪 Q: do you have an 819 flight from DENVER to SAN FRANCISCO
🔧 SQL_PRED: SELECT * FROM FLIGHT AS FLIGHTalias0 WHERE FLIGHTalias0.FLIGHT_ID = 819 AND FLIGHTalias0.TO_AIRPORT = ( SELECT AIRPORT_CODE FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 WHERE CITYalias0.CITY_NAME = ""SAN FRANCISCO"" AND CITYalias0.CITY_CODE = ( SELECT CITY_CODE FROM CITY AS CITYalias0 WHERE CITYalias0.CITY_NAME = ""DENVER"" ) ) ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""SAN FRANCISCO"" AND FLIGHTalias0.DEPARTURE_TIME = 819 AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""DENVER"" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE ;
✅ MATCH: 

 67%|██████▋   | 2/3 [00:08<00:04,  4.66s/it]

🧪 Q: i would like a flight from KANSAS CITY to ST. PAUL departing at 1600
🔧 SQL_PRED: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, CITY AS CITYalias0, CITY AS CITYalias1, DATE_DAY AS DATE_DAYalias0, DAYS AS DAYSalias0, FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""ST. PAUL"" AND DATE_DAYalias0.DAY_NUMBER = 1 AND DATE_DAYalias0.MONTH_NUMBER = 4 AND DATE_DAYalias0.YEAR = 1991 AND DAYSalias0.DAY_NAME = DATE_DAYalias0.DAY_NAME AND FLIGHTalias0.FLIGHT_DAYS = DAYSalias0.DAYS_CODE AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.DEPARTURE_TIME = 1600 ) ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1

100%|██████████| 3/3 [00:14<00:00,  4.80s/it]

🧪 Q: do you have a flight leaving BOSTON at 645 going to WASHINGTON
🔧 SQL_PRED: SELECT * FROM FLIGHT AS FLIGHTalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , CITY AS CITYalias0 , CITY AS CITYalias1 WHERE ( ( CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = "WASHINGTON" AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = "BOSTON" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE AND FLIGHTalias0.DEPARTURE_TIME = 645 ) ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""WASHINGTON"" AND FLIGHTalias0.DEPARTURE_TIME = 645 AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CIT